# Model training and evaluation — Milestone 4

This notebook compares a majority baseline, Logistic Regression, Random Forest, and Histogram Gradient Boosting. It uses a chronological holdout and five expanding-window time-series folds inside the training period. Model selection is based on mean cross-validation Average Precision; the final test period is evaluated once after selection.

A source audit found that the supplied columns named `LAGGED_PRECIPITATION` and `LAGGED_AVG_WIND_SPEED` match seven-day rolling statistics that include the current date. The model therefore rebuilds both predictors from prior rows only.

**Important scope:** this remains same-day historical classification because daily weather from the target date is included; it is not a validated early-warning system.

## 1. Run the reproducible modeling pipeline

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / "run_milestone4.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from run_milestone4 import run
summary = run()

## 2. Compare chronological cross-validation scores

In [ ]:
import pandas as pd
comparison = pd.read_csv(ROOT / "reports" / "model_comparison_cv.csv")
comparison

## 3. Inspect held-out metrics and predictions

In [ ]:
import json
metrics = json.loads((ROOT / "reports" / "final_model_metrics.json").read_text())
print("Selected model:", metrics["selected_model"])
print("Test metrics:")
print(pd.Series(metrics["test_metrics"]).round(3))
print("Confusion matrix [true 0/1 x predicted 0/1]:")
print(pd.DataFrame(metrics["confusion_matrix_labels_0_1"], index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))

## 4. Review saved predictions and feature diagnostics

In [ ]:
predictions = pd.read_csv(ROOT / "data" / "model_ready" / "test_predictions.csv")
importance = pd.read_csv(ROOT / "reports" / "selected_model_permutation_importance.csv")
print(predictions.head())
importance.head(10)

## Interpretation checklist

- Compare CV mean scores and their fold-to-fold standard deviations; do not report only one score.
- Average Precision summarizes the precision–recall ranking and is the primary model-selection metric here.
- Accuracy alone can hide errors on either class, so review balanced accuracy, precision, recall, F1, ROC AUC, and the confusion matrix.
- The 2016–2023 test period is later than the training period and has a higher positive-label rate; temporal distribution shift limits direct generalization claims.
- Permutation importance is a diagnostic on the held-out sample, not evidence of causation.
- Do not present this project as operational advance warning without time-safe predictors, independent validation, and domain review.